<a href="https://colab.research.google.com/github/TRPG-masterbot/data/blob/main/%EC%9E%A0%EA%B8%B4%EC%A2%85_%ED%86%B5%ED%95%A9_test_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 잠긴 종 통합 v2 (rule_rag.py + sce_rag.py + 분류기)

두 RAG는 **각자의 파이썬 파일**로 두고, 이 노트북은 `import`로 가져와 **분류기 + 한 턴 진행 + 시험**만 합니다.

```
rule_rag.py   (Min)      룰북 RAG
sce_rag.py    (Kim)      시나리오 RAG
이 노트북     (통합)      분류기 → 규칙 / 시나리오 / 둘 다 → 답
```

```
플레이어 입력
   ↓ classify (14B가 '규칙' / '시나리오' / '둘 다' 중 하나를 고름)
   ├ 규칙     → rule_rag.answer          → 📖 규칙 설명
   ├ 시나리오 → sce_rag.search + 서술     → 📜 장면
   └ 둘 다    → 위 두 가지를 모두         → 📖 + 📜
```

## 두 파일이 지켜야 할 약속 (함수 이름과 입출력)

| 파일 | 함수 | 하는 일 | 받는 것 → 돌려주는 것 |
|---|---|---|---|
| `rule_rag.py` | `load(url)` | 룰북 데이터 받기 + 임베딩 + 검색 준비 (import 때가 아니라 이 함수를 불렀을 때 실행) | 룰북 파일 주소 → 없음 |
| | `answer(question, attempts, show_orig)` | 질문 → 검색 → 근거 선택 → 설명 | 질문 → `{"status": "ok"/"refuse"/"hidden", "explain": 글, "title": 규칙 제목, "warn": [..]}` |
| `sce_rag.py` | `load()` | cards.json 읽기 + Chroma 저장 | 없음 → 없음 |
| | `make_state(scene, cls)` | 시험용 상태 만들기 (실제 게임에서는 엔진이 만들어 줌) | 장면 번호, 직업 → state |
| | `search(question, state, k)` | 지금 볼 수 있는 카드만 검색 | 글, state, 개수 → `[(카드 id, 유사도), ...]` |
| | `card_text(card_id)` | 카드의 플레이어용 글 | 카드 id → 글 |

- `rule_rag.answer`는 Min이 이미 가진 함수 그대로입니다. 파일로 옮기고 `load()`만 만들면 됩니다.
- 두 파일 모두 **Ollama가 켜진 뒤에** `load()`를 부릅니다.
- 룰북·카드 데이터는 구글 드라이브가 아니라 GitHub에서 받습니다.

| 목차 | 하는 일 |
|---|---|
| L0 | 설치, Ollama, 모델 준비 |
| L1 | 두 파일 내려받아 import |
| L2 | 14B 호출 + 시나리오 서술 |
| L3 | 입력 분류기 (규칙 / 시나리오 / 둘 다) |
| L4 | 한 턴 `game_turn` |
| L5 | 시험 |

# L0. 설치와 모델 준비

In [ ]:
# ===== ##0 설치와 import (런타임을 바꾸면 이 셀부터 다시 실행) =====
!pip install -q chromadb faiss-cpu sentence-transformers

import os, re, sys, json, time, math, subprocess, importlib
import requests

# GPU 확인 (없으면 14B가 매우 느립니다)
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print("GPU :", gpu or "없음 (런타임을 GPU로 바꾸세요)")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
t

In [ ]:
# ===== ##0-b Ollama 설치 =====
# 다운로드가 중간에 끊기면 설치가 깨지므로, 끊기면 다시 받는다 (최대 4번).
!apt-get install -y -qq zstd

for attempt in range(1, 5):
    print(f"--- Ollama 설치 시도 {attempt}/4 ---")
    result = subprocess.run("curl -fsSL http://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
    ok = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    if result.returncode == 0 and "Unexpected EOF" not in result.stderr + result.stdout and os.path.exists("/usr/local/bin/ollama"):
        print("설치 완료:", (ok.stdout or ok.stderr).strip().splitlines()[-1])
        break
    print("설치가 중간에 끊겼어요. 다시 시도합니다.")
    time.sleep(5)
else:
    raise RuntimeError("Ollama 설치가 계속 실패했어요. 잠시 뒤 이 셀을 다시 실행하세요.")

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
--- Ollama 설치 시도 1/4 ---
설치 완료: Warning: client version is 0.40.16


In [ ]:
# ===== ##0-c Ollama 서버 켜고 모델 받기 (몇 분 걸림) =====
OLLAMA_URL = "http://localhost:11434"
GEN_MODEL = "qwen2.5:14b"        # 분류·번역·설명·서술을 모두 이 모델이 함
EMBED_MODEL = "bge-m3"           # 룰북 검색용 임베딩
os.environ["OLLAMA_KEEP_ALIVE"] = "1h"
subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)

for _ in range(120):
    try:
        if requests.get(OLLAMA_URL, timeout=2).status_code == 200:
            print("서버 준비 완료")
            break
    except Exception:
        pass
    time.sleep(1)

for m in [EMBED_MODEL, GEN_MODEL]:
    print("받는 중:", m)
    subprocess.run(["ollama", "pull", m], check=True)
print("모델 준비 완료")

서버 준비 완료
받는 중: bge-m3
받는 중: qwen2.5:14b
모델 준비 완료


# L1. 두 파일 내려받아 import

GitHub에 올라온 `rule_rag.py`(Min), `sce_rag.py`(Kim)를 받아서 `import` 합니다.
파일 위치가 다르면 아래 두 주소만 고치세요.

In [ ]:
# ===== ##1 rule_rag.py, sce_rag.py 받아서 import =====
BASE = "https://raw.githubusercontent.com/TRPG-masterbot/data/main"
MODULES = {
    "rule_rag.py": f"{BASE}/Min/rule_rag.py",     # 룰북 RAG (Min)
    "sce_rag.py":  f"{BASE}/Kim/sce_rag.py",      # 시나리오 RAG (Kim)
}

RULES_URL = f"{BASE}/Min/rules_new.jsonl"       # 룰북 청크 파일 (rule_rag.py 안의 주소가 달라서 여기서 넘겨 줌)

os.makedirs("modules", exist_ok=True)
for name, url in MODULES.items():
    r = requests.get(url, timeout=60)
    if r.status_code == 404:                    # GitHub에 아직 안 올라온 파일은 직접 올리기
        from google.colab import files
        print(f"GitHub에 {name}이 아직 없어요. 내 컴퓨터에서 직접 올려 주세요:")
        files.upload()                          # 파일 선택 창이 뜸
        os.replace(name, f"modules/{name}")
        print("직접 올림:", name)
        continue
    r.raise_for_status()
    open(f"modules/{name}", "wb").write(r.content)
    print("받음:", name, round(len(r.content) / 1000, 1), "KB")

if "modules" not in sys.path:
    sys.path.insert(0, "modules")

import rule_rag, sce_rag
importlib.reload(rule_rag)         # 파일을 새로 받았을 때 바뀐 내용을 다시 읽기
importlib.reload(sce_rag)

rule_rag.load(url=RULES_URL)       # 룰북 데이터 + 임베딩 + Chroma (처음엔 몇 분 걸림)
sce_rag.load()                     # 카드 읽기 + Chroma 저장
print("두 RAG 준비 완료")

받음: rule_rag.py 37.9 KB
받음: sce_rag.py 5.5 KB
룰북 받음: rules_new.jsonl
검색 대상 청크: 2180


  0%|          | 0/69 [00:00<?, ?it/s]

임베딩 새로 계산: (2180, 1024)
준비 완료


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

sce_rag 준비 완료: 카드 166장 중 검색 대상 112장
두 RAG 준비 완료


# L2. 14B 호출 + 시나리오 서술

분류와 서술은 룰북과 같은 14B(Ollama)로 합니다.
서술은 검색된 카드 글만 근거로 쓰게 하고, **큰따옴표 대사는 카드에 적힌 문장만** 쓰게 했습니다.

In [ ]:
# ===== ##2 14B에게 질문 보내고 답 받기 =====
def ask_llm(messages, max_new_tokens=300, seed=42, temperature=0.0):
    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": GEN_MODEL,
              "messages": messages,
              "stream": False,
              "options": {"temperature": temperature,   # 0 = 같은 입력 -> 같은 출력 (재시도할 때는 올린다)
                          "seed": seed,
                          "num_ctx": rule_rag.NUM_CTX,   # 룰북 RAG와 같은 값(4096). 값이 다르면 Ollama가 14B를 다시 올려서 느려진다
                          "num_predict": max_new_tokens}},   # 출력 길이 제한
        timeout=600,
    )
    res.raise_for_status()
    return res.json()["message"]["content"].strip()

In [ ]:
# ===== ##3 서술 결과 검사 (코드) =====

# 한글, 영어·숫자(ASCII), 공백, 기본 기호 이외의 글자가 있으면 깨진 글자 (한자·일본어·태국어 등)
BROKEN = re.compile(r"[^\uac00-\ud7a3\u3131-\u318e\x00-\x7f\s…·“”‘’—–]")

# 카드에 없으면 지어낸 것으로 보는 표정·몸짓 단어
GESTURE_WORDS = ["고개", "미소", "웃음", "눈빛", "표정", "한숨", "눈썹", "눈살", "찌푸", "생각에 잠",
                 "경계하", "당황", "시선", "손을 뻗", "몸을 피", "몸을 돌", "몸을 조금", "입가", "어깨를",
                 "눈을 크게", "눈을 감", "침묵", "천장"]

QUOTE = re.compile(r"[\"“]([^\"”]+)[\"”]")           # 큰따옴표 안의 글 (대사)

def squash(s):
    # 비교용: 공백·문장부호·따옴표를 지움
    return re.sub(r"[\s.,!?…·\"'“”‘’]", "", s)

def narration_problems(answer, card_texts, check_length=True):
    problems = []
    source = "\n".join(card_texts)

    if BROKEN.search(answer):
        problems.append("한국어·영어 이외의 글자가 섞임")

    for quote in QUOTE.findall(answer):                # 큰따옴표 대사는 카드에 있는 문장만
        if squash(quote) not in squash(source):
            problems.append(f"카드에 없는 대사: \"{quote[:25]}\"")

    for word in GESTURE_WORDS:                         # 카드에 없는 표정·몸짓
        if word in answer and word not in source:
            problems.append(f"카드에 없는 표정·몸짓: {word}")

    if check_length and len(answer.strip()) < 20:
        problems.append("너무 짧음")
    return problems

def fix_name(text):
    # '플레이어'를 '당신'으로 바꾸되 뒤에 붙는 조사도 맞춘다 (플레이어는 -> 당신은)
    particles = {"는": "은", "가": "이", "를": "을", "의": "의", "와": "과", "도": "도", "에게": "에게"}
    def swap(m):
        return "당신" + particles.get(m.group(2), m.group(2))
    text = re.sub(r"(플레이어|주인공)(는|가|를|의|와|도|에게)", swap, text)
    return re.sub(r"플레이어|주인공", "당신", text)

def clean_narration(answer, card_texts):
    # 끝까지 문제가 남았을 때: 문제 있는 대사·문장을 빼고, 남는 게 없으면 1등 카드 글을 그대로 쓴다
    source_squashed = squash("\n".join(card_texts))
    # 1) 카드에 없는 큰따옴표 대사를 통째로 지운다 (대사 안의 마침표로 문장이 쪼개지기 전에 먼저 처리)
    answer = QUOTE.sub(lambda m: m.group(0) if squash(m.group(1)) in source_squashed else "", answer)
    # 2) 남은 글에서 문제 있는 문장을 뺀다
    sentences = re.split(r"(?<=[.!?…])\s+|\n+", answer)
    kept = [s for s in sentences if s and not narration_problems(s, card_texts, check_length=False)]
    text = " ".join(kept).strip()
    if len(text) < 40:                                  # 남은 글이 짧아 정보가 없으면 1등 카드 글을 덧붙인다
        base = card_texts[0] if card_texts else "주변은 고요하다."
        text = (text + " " + base).strip()
    return text

In [ ]:
# ===== ##4 시나리오 서술 (프롬프트 + 코드 검사 + 다시 쓰기) =====
SYS_NARRATE = """너는 한국어 1인용 TRPG의 진행자(마스터)다. 플레이어의 입력을 읽고 무엇을 하려는지 먼저 파악한 뒤, 그 행동의 결과를 이어서 들려준다.

[판단 기준]
- 행동이 [이번에 드러난 정보]와 [규칙]에서 크게 벗어나지 않으면, 이야기가 자연스럽게 흘러가도록 결과를 네가 판단해서 묘사한다.
- 정보로 답할 수 없는 행동이면 억지로 지어내지 말고 알아낼 수 있는 것이 없다고 짧게 말한다.
- 결과가 불확실하거나 위험한 행동은 시도하는 모습까지만 쓰고 결과를 확정하지 않는다. (판정은 게임이 한다)

[반드시 지킬 것]
1. 정보에 적힌 사실과 반대로 쓰지 않는다. 정보에 없는 인물, 장소, 물건, 사건을 새로 만들지 않는다.
2. 큰따옴표 대사는 [사용 가능한 대사]에 있는 문장을 그대로 옮길 때만 쓴다. 거기 없는 말을 NPC가 해야 하면 큰따옴표 없이 간접화법으로 쓴다. (예: 촌장은 마지못해 인정한다.)
3. 정보에 없는 표정, 눈빛, 몸짓, 감정, 침묵 같은 동작을 쓰지 않는다.
4. 플레이어를 '당신'이라고 부른다. 당신의 대사는 큰따옴표로 새로 만들지 않고, 당신의 말·생각·감정도 대신 정하지 않는다.
5. 2~4문장, 현재형으로 쓴다. 다음에 무엇을 하라고 지시하지 않는다. 한국어만 쓴다."""

def narrate_scene(text, state, k=2, rule_text=None, tries=3):
    hits = sce_rag.search(text, state, k=k)                        # [(카드 id, 유사도), ...]
    card_texts = [sce_rag.card_text(cid) for cid, sim in hits]
    shown = "\n".join(f"- {t}" for t in card_texts)

    # 카드에 적힌 큰따옴표 대사를 미리 뽑아 준다 (쓸 수 있는 대사를 알려 주면 새로 지어낼 필요가 줄어든다)
    quotes = [q for t in card_texts for q in QUOTE.findall(t)]
    quote_block = "\n".join(f'- "{q}"' for q in quotes) if quotes else "(없음: 대사를 쓰지 말고 간접화법으로 쓴다)"

    user = f"[플레이어 입력]\n{text}\n\n[이번에 드러난 정보]\n{shown}\n\n[사용 가능한 대사]\n{quote_block}\n"
    if rule_text:
        user += f"\n[규칙]\n{rule_text}\n"
    user += "\n위 정보만 바탕으로 장면을 묘사해 줘."
    messages = [{"role": "system", "content": SYS_NARRATE}, {"role": "user", "content": user}]

    answer, problems = "", []
    for t in range(tries):
        answer = ask_llm(messages, max_new_tokens=300, seed=42 + t, temperature=0.0 if t == 0 else 0.5)
        problems = narration_problems(answer, card_texts)
        if not problems:
            break
        # 문제를 알려 주고 다시 쓰게 한다 (온도 0에서는 같은 답이 나오므로 내용으로 고치게 함)
        messages = messages[:2] + [
            {"role": "assistant", "content": answer},
            {"role": "user", "content": "다음 문제가 있어. 고쳐서 다시 써 줘: " + "; ".join(problems)},
        ]

    fallback = False
    if problems:                                                   # 3번 다 실패
        answer = clean_narration(answer, card_texts)
        fallback = True
    answer = fix_name(answer)                                      # 호칭 통일 (조사도 맞춤)

    info = {"tries": t + 1, "problems": problems, "fallback": fallback}
    return hits, answer, info

In [ ]:
# ===== ##4-a 룰북 질문 -> 설명 (rule_rag.answer 감싸기) =====
RULE_ATTEMPTS = 1      # 설명이 검사에 걸렸을 때 다시 쓰는 최대 횟수. 1 = 다시 안 씀 (빠름)

def ask_rule(question):
    a = rule_rag.answer(question, attempts=RULE_ATTEMPTS, show_orig=False)
    if a["status"] == "ok":
        return a["explain"], a
    return "이 질문에 맞는 규칙을 확실히 찾지 못했어요. 질문을 조금 바꿔서 다시 물어봐 주세요.", a

# L3. 입력 분류기

14B에게 예시를 보여 주고 `규칙` / `시나리오` / `둘 다` 중 하나만 고르게 합니다.
- `둘 다`: 게임 속 행동을 하면서 그 행동의 규칙(판정 방법 등)도 궁금한 입력
- `?`나 `규칙:`으로 시작하면 분류 없이 규칙으로 보냅니다.
- 애매하면 `시나리오`로 보냅니다.

In [ ]:
# ===== ##5 입력 분류기 =====
INTENT_PROMPT = """플레이어가 입력한 말이 어떤 종류인지 하나만 고르세요.
- 규칙: 룰북에 일반적으로 적힌 내용을 묻는 질문. 판정 방법, 상태의 효과, 행동의 종류, 장비·물건의 가격과 성능처럼 게임 밖에서도 똑같이 답할 수 있는 것
- 시나리오: 지금 게임 속에서 하는 행동이나 대화, 또는 '여기·지금·이·저'처럼 이야기 속 장소·인물·물건·사건에 대해 묻는 말
- 둘 다: 게임 속에서 어떤 행동을 하려고 하면서, 그 행동에 필요한 규칙(판정 방법 등)도 함께 묻는 말
'규칙', '시나리오', '둘 다' 중 하나만 쓰세요.

입력: 치명타가 나오면 피해를 어떻게 계산해?
답: 규칙
입력: 촌장에게 실종자에 대해 묻는다
답: 시나리오
입력: 밧줄 50피트는 얼마야?
답: 규칙
입력: 우물 안을 들여다본다
답: 시나리오
입력: 우물 벽을 타고 내려가고 싶은데 무슨 판정을 해야 해?
답: 둘 다
입력: 저 노인에게 말을 걸어 본다
답: 시나리오
입력: 촌장 서재 문을 몰래 열어 보려는데 어떤 판정이야?
답: 둘 다

입력: {text}
답:"""

def classify(text):
    if re.match(r"\s*(규칙|룰)\s*[:：]", text) or text.strip().startswith("?"):
        return "규칙"                                   # 강제 규칙 질문
    out = ask_llm([{"role": "user", "content": INTENT_PROMPT.format(text=text)}], max_new_tokens=6)
    if "둘" in out:
        return "둘 다"
    if "규칙" in out:
        return "규칙"
    return "시나리오"                                    # 애매하면 시나리오

# L4. 통합 한 턴 `game_turn`

`state`(장면, 플래그, 방문한 방, 직업)는 실제 게임에서는 게임 엔진이 만들어 넘겨줍니다.

In [ ]:
# ===== ##6 통합 한 턴 =====
def game_turn(text, state):
    t0 = time.time()
    kind = classify(text)
    parts, sources, warn, narration_info = [], {}, [], None
    rule_ok_text = None

    if kind in ("규칙", "둘 다"):
        question = re.sub(r"^\s*(규칙|룰)\s*[:：]\s*|^\s*\?\s*", "", text)     # '규칙:' / '?' 머리말 제거
        rule_text, info = ask_rule(question)
        parts.append(("📖 규칙", rule_text))
        sources["rule"] = info.get("title")
        warn += info.get("warn") or []
        if info.get("status") == "ok":
            rule_ok_text = rule_text

    if kind in ("시나리오", "둘 다"):
        hits, narration, narration_info = narrate_scene(text, state, rule_text=rule_ok_text)
        parts.append(("📜 장면", narration))
        sources["scenario"] = [cid for cid, sim in hits]

    shown = parts[0][1] if len(parts) == 1 else "\n\n".join(f"{head}\n{body}" for head, body in parts)
    return {"kind": kind, "text": shown, "sources": sources, "warn": warn,
            "narration_info": narration_info, "seconds": round(time.time() - t0, 1)}

def show_turn(text, state):
    r = game_turn(text, state)
    print(f"[입력] {text}")
    print(f"[분류] {r['kind']} | 근거: {r['sources']} | {r['seconds']}초")
    if r["warn"]:
        print(f"[경고] {r['warn']}")
    ni = r["narration_info"]
    if ni and (ni["tries"] > 1 or ni["fallback"]):
        print(f"[서술 검사] {ni['tries']}번 만에 통과 못함" if ni["fallback"] else f"[서술 검사] {ni['tries']}번째에 통과",
              "| 마지막 문제:", ni["problems"] if ni["fallback"] else "(해결됨)")
    print(f"[답] {r['text']}")
    print("-" * 40)
    return r

# L5. 시험

In [ ]:
# ===== ##7 예시 입력 (시나리오 / 규칙 / 둘 다를 섞어서) =====
SCENE = 1
CLS = "fighter"      # fighter / cleric / rogue / wizard
st = sce_rag.make_state(SCENE, CLS)

samples = [
    "촌장에게 마을에서 있었던 일을 물어본다",                  # 시나리오 일 것
    "넘어졌을 때 일어나려면 이동력이 얼마나 들어?",            # 규칙 일 것
    "우물 벽을 타고 내려가고 싶은데 무슨 판정을 해야 해?",     # 둘 다 일 것
    "눈이 안 보이는 할머니에게 말을 걸어 본다",               # 시나리오 일 것
]
for t in samples:
    show_turn(t, st)

[입력] 촌장에게 마을에서 있었던 일을 물어본다
[분류] 시나리오 | 근거: {'scenario': ['scn-v8-rumor-halvik', 'scn-s1-npc-halvik']} | 51.4초
[서술 검사] 3번 만에 통과 못함 | 마지막 문제: ['카드에 없는 대사: "마을에서는 크게 특이한 일이 없소. 다만 우물"']
[답] 당신은 촌장에게 마을에서 있었던 일을 물어본다. 촌장은 잠시 침묵하고는 천장을 바라본다. "마을에서는 크게 특이한 일이 없소. 다만 우물가에서 종소리 같은 소리가 자주 들린다는 것뿐이오."
----------------------------------------
[입력] 넘어졌을 때 일어나려면 이동력이 얼마나 들어?
[분류] 규칙 | 근거: {'rule': 'Prone [Condition]'} | 48.0초
[답] 넘어진 상태에서는 일어나려면 자신의 속도의 절반을 이동력으로 소모해야 합니다. (내림)
----------------------------------------
[입력] 우물 벽을 타고 내려가고 싶은데 무슨 판정을 해야 해?
[분류] 둘 다 | 근거: {'rule': 'Climbing', 'scenario': ['scn-v8-well-pulley', 'scn-s1-clue-well']} | 122.2초
[답] 📖 규칙
DM의 선택에 따라 미끄러운 표면이나 손잡이가 적은 표면을 등반할 때 성공적인 DC 15 근력(체조) 판정이 필요할 수 있습니다.

📜 장면
당신은 우물 벽을 타고 내려가려고 한다. 도르래 틀이 있는 곳에서 밧줄을 설치하고, 우물 뚜껑을 열어 사다리 디딤쇠를 확인한 뒤, 천천히 밧줄을 잡고 내려가기 시작한다.
----------------------------------------
[입력] 눈이 안 보이는 할머니에게 말을 걸어 본다
[분류] 시나리오 | 근거: {'scenario': ['scn-q4-npc-mera-song', 'scn-s1-npc-mera']} | 17.6초
[서술 검사] 3번 만에 통과 

In [ ]:
# ===== ##8 직접 입력해 보기 (빈 줄 입력하면 끝) =====
# 규칙 질문은 앞에 '?'를 붙이면 분류 없이 룰북 RAG로 갑니다.
SCENE = 1
CLS = "fighter"
st = sce_rag.make_state(SCENE, CLS)

while True:
    text = input("\n[입력] > ").strip()
    if not text:
        break
    show_turn(text, st)

# [입력] > 할빅을 때려본다
# [입력] 할빅을 때려본다
# [분류] 둘 다 | 근거: {'rule': 'Attack [Action]', 'scenario': ['scn-s1-npc-halvik', 'scn-g-halvik-house']} | 89.0초
# [답] 📖 규칙
# 공격 행동을 취할 때, 무기를 사용하거나 맨손 타격으로 공격 굴림을 한 번 할 수 있습니다.

# 📜 장면
# 할빅은 넓은 방 한가운데 서서 당신을 경계하는 눈빛으로 바라본다. 당신이 그를 향해 손을 뻗자, 할빅은 빠르게 몸을 피하며 "이런, 뭐 하는 거요?"라고 말한다.
# ----------------------------------------

# [입력] > 할빅에게 선금은 필요없다고 말하고 의뢰를 받아들인다
# [입력] 할빅에게 선금은 필요없다고 말하고 의뢰를 받아들인다
# [분류] 시나리오 | 근거: {'scenario': ['scn-s1-npc-halvik', 'scn-g-halvik-house']} | 7.4초
# [답] 할빅은 선금을 내민 손을 천천히 거두며 고개를 끄덕인다. "선금은 필요없다구? 좋소. 그럼 의뢰를 받아주시오." 그는 다시 우물을 되살려달라는 의뢰 내용을 상기시킨다. "60피트쯤 내려가면 물이 새는 틈이 있소. 거기만 막아 주면 되오."
# ----------------------------------------

# [입력] > 할빅에 건넨 선금을 무시한채 수상한 단서가 없는지 방안을 뒤져본다
# [입력] 할빅에 건넨 선금을 무시한채 수상한 단서가 없는지 방안을 뒤져본다
# [분류] 둘 다 | 근거: {'rule': 'Search [Action]', 'scenario': ['scn-s1-npc-halvik', 'scn-g-halvik-house']} | 89.2초
# [답] 📖 규칙
# 수색 행동을 취하면, 눈에 띄지 않는 것을 알아내기 위해 지혜 판정을 합니다. 수색 테이블은 당신이 감지하려는 것이 무엇인지에 따라 적용 가능한 기술을 제안합니다. 예를 들어, 의심스러운 단서를 찾으려면 인사이트(지혜) 기술을 사용할 수 있습니다.

# 📜 장면
# 플레이어가 방 안을 뒤지지만 특이한 단서는 없다. 벽난로 위에 걸린 낡은 석판만 눈에 들어오고, 서재로 가는 문은 반쯤 열려 있지만 그 안쪽은 볼 수 없다.

# [입력] > 할빅에게 선금을 더 줄 순 없냐고 물어본다
# [입력] 할빅에게 선금을 더 줄 순 없냐고 물어본다
# [분류] 시나리오 | 근거: {'scenario': ['scn-s1-npc-halvik', 'scn-g-halvik-house']} | 56.4초
# [서술 검사] 3번 만에 통과 못함 | 마지막 문제: ['카드에 없는 대사: "촌장님, 선금을 좀 더 드려도 될까요?"', '카드에 없는 대사: "우물 작업은 그대로 진행하면 되오. 추가 비용"']
# [답] 당신은 할빅에게 다가가며 말을 건넨다. 할빅은 잠시 고민하는 척하다가 말한다.
# ----------------------------------------

# [입력] > 우물가 아래를 쳐다본다
# [입력] 우물가 아래를 쳐다본다
# [분류] 시나리오 | 근거: {'scenario': ['scn-s1-clue-well', 'scn-s1-npc-mera']} | 6.6초
# [답] 당신이 우물 아래를 내려다본다. 바닥까지는 멀지만, 사다리 디딤쇠가 어둠 속으로 이어진다. 최근에 누군가 이 우물을 들여다본 흔적이 역력하다. 메라의 말대로 이 우물 아래에는 옛 수도원이 있었다고 한다.
# ----------------------------------------

# [입력] > 밧줄없이 우물가 아래로 내려갈 방법이 없는지 찾아본다
# [입력] 밧줄없이 우물가 아래로 내려갈 방법이 없는지 찾아본다
# [분류] 시나리오 | 근거: {'scenario': ['scn-v8-well-pulley', 'scn-s1-clue-well']} | 7.7초
# [답] 채석장 시절의 도르래 틀이 있는 곳을 다시 살펴보며, 우물가 주변을 뒤지기 시작한다. 그러나 주변에는 긴 밧줄이나 대체할 만한 줄 같은 것은 보이지 않는다. 도르래 틀은 밧줄을 걸 수 있을 만한 상태이지만, 현재는 아무것도 걸려 있지 않다.
# ----------------------------------------



[입력] > 할빅에게 선금을 더 줄 순 없냐고 물어본다
[입력] 할빅에게 선금을 더 줄 순 없냐고 물어본다
[분류] 시나리오 | 근거: {'scenario': ['scn-s1-npc-halvik', 'scn-g-halvik-house']} | 56.4초
[서술 검사] 3번 만에 통과 못함 | 마지막 문제: ['카드에 없는 대사: "촌장님, 선금을 좀 더 드려도 될까요?"', '카드에 없는 대사: "우물 작업은 그대로 진행하면 되오. 추가 비용"']
[답] 당신은 할빅에게 다가가며 말을 건넨다. 할빅은 잠시 고민하는 척하다가 말한다.
----------------------------------------

[입력] > 우물가 아래를 쳐다본다
[입력] 우물가 아래를 쳐다본다
[분류] 시나리오 | 근거: {'scenario': ['scn-s1-clue-well', 'scn-s1-npc-mera']} | 6.6초
[답] 당신이 우물 아래를 내려다본다. 바닥까지는 멀지만, 사다리 디딤쇠가 어둠 속으로 이어진다. 최근에 누군가 이 우물을 들여다본 흔적이 역력하다. 메라의 말대로 이 우물 아래에는 옛 수도원이 있었다고 한다.
----------------------------------------

[입력] > 밧줄없이 우물가 아래로 내려갈 방법이 없는지 찾아본다
[입력] 밧줄없이 우물가 아래로 내려갈 방법이 없는지 찾아본다
[분류] 시나리오 | 근거: {'scenario': ['scn-v8-well-pulley', 'scn-s1-clue-well']} | 7.7초
[답] 채석장 시절의 도르래 틀이 있는 곳을 다시 살펴보며, 우물가 주변을 뒤지기 시작한다. 그러나 주변에는 긴 밧줄이나 대체할 만한 줄 같은 것은 보이지 않는다. 도르래 틀은 밧줄을 걸 수 있을 만한 상태이지만, 현재는 아무것도 걸려 있지 않다.
----------------------------------------

[입력] > 


In [ ]:
# 서버를 노트북과 분리된 별도 그룹으로 켜기 (정지 버튼을 눌러도 같이 죽지 않게)
log = open("ollama.log", "w")
subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log, start_new_session=True)

for _ in range(60):
    try:
        if requests.get(OLLAMA_URL, timeout=2).status_code == 200:
            print("서버 준비 완료"); break
    except Exception:
        pass
    time.sleep(1)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

서버 준비 완료
NAME             ID              SIZE      MODIFIED       
qwen2.5:14b      7cdf5a0187d5    9.0 GB    51 minutes ago    
bge-m3:latest    790764642607    1.2 GB    53 minutes ago    

